### Needs `01-ReferenceDFs.ipynb` to be executed first

Reads from:
- `gs_extractions_raw.json`
- `gs_extractions_raw_ynorm.json`
- `year_scope_df.json`

# Preperations

In [42]:
import pandas as pd
import numpy as np

## Import Data

In [43]:
gs_extractions_raw  = pd.read_json("gs_extractions_raw.json")
gs_extractions_raw_ynorm = pd.read_json("gs_extractions_raw_ynorm.json")
year_scope_df       = pd.read_json("year_scope_df.json")

CATEGORIES = ["value", "unit"]
VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2"]

#### Cleanup Import

In [44]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    gs_extractions_raw[col] = gs_extractions_raw[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    gs_extractions_raw_ynorm[col] = gs_extractions_raw_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [45]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set


def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## Straight Forward

In [46]:
just_eval, expended = eval(gs_extractions_raw, False)

print_eval(just_eval)

### value ###
t1: True = 2078 || False = 130 || Quota = 94.11%
t2: True = 2087 || False = 121 || Quota = 94.52%
m1: True = 1607 || False = 601 || Quota = 72.78%
m2: True = 1923 || False = 285 || Quota = 87.09%
i1: True = 1874 || False = 334 || Quota = 84.87%
i2: True = 1858 || False = 350 || Quota = 84.15%

### unit ###
t1: True = 1952 || False = 256 || Quota = 88.41%
t2: True = 1935 || False = 273 || Quota = 87.64%
m1: True = 1764 || False = 444 || Quota = 79.89%
m2: True = 1787 || False = 421 || Quota = 80.93%
i1: True = 1761 || False = 447 || Quota = 79.76%
i2: True = 1744 || False = 464 || Quota = 78.99%



### Exact

In [49]:
just_eval_exact, _ = eval(gs_extractions_raw, True)

print_eval(just_eval_exact)

### value ###
t1: True = 2057 || False = 151 || Quota = 93.16%
t2: True = 2056 || False = 152 || Quota = 93.12%
m1: True = 1607 || False = 601 || Quota = 72.78%
m2: True = 1909 || False = 299 || Quota = 86.46%
i1: True = 1851 || False = 357 || Quota = 83.83%
i2: True = 1820 || False = 388 || Quota = 82.43%

### unit ###
t1: True = 1943 || False = 265 || Quota = 88.0%
t2: True = 1926 || False = 282 || Quota = 87.23%
m1: True = 1755 || False = 453 || Quota = 79.48%
m2: True = 1785 || False = 423 || Quota = 80.84%
i1: True = 1755 || False = 453 || Quota = 79.48%
i2: True = 1735 || False = 473 || Quota = 78.58%



## YNROM

In [50]:
ynorm, expended_ynorm = eval(gs_extractions_raw_ynorm, False)

print_eval(ynorm)

### value ###
t1: True = 2084 || False = 124 || Quota = 94.38%
t2: True = 2100 || False = 108 || Quota = 95.11%
m1: True = 1587 || False = 621 || Quota = 71.88%
m2: True = 1920 || False = 288 || Quota = 86.96%
i1: True = 1884 || False = 324 || Quota = 85.33%
i2: True = 1870 || False = 338 || Quota = 84.69%

### unit ###
t1: True = 1965 || False = 243 || Quota = 88.99%
t2: True = 1956 || False = 252 || Quota = 88.59%
m1: True = 1755 || False = 453 || Quota = 79.48%
m2: True = 1786 || False = 422 || Quota = 80.89%
i1: True = 1777 || False = 431 || Quota = 80.48%
i2: True = 1760 || False = 448 || Quota = 79.71%



### Exact

In [51]:
ynorm_exact, _ = eval(gs_extractions_raw_ynorm, True)

print_eval(ynorm_exact)

### value ###
t1: True = 2063 || False = 145 || Quota = 93.43%
t2: True = 2069 || False = 139 || Quota = 93.7%
m1: True = 1587 || False = 621 || Quota = 71.88%
m2: True = 1906 || False = 302 || Quota = 86.32%
i1: True = 1845 || False = 363 || Quota = 83.56%
i2: True = 1816 || False = 392 || Quota = 82.25%

### unit ###
t1: True = 1956 || False = 252 || Quota = 88.59%
t2: True = 1947 || False = 261 || Quota = 88.18%
m1: True = 1746 || False = 462 || Quota = 79.08%
m2: True = 1784 || False = 424 || Quota = 80.8%
i1: True = 1771 || False = 437 || Quota = 80.21%
i2: True = 1751 || False = 457 || Quota = 79.3%



# Detailed Comparison

In [52]:
def save_eval_comparison(eval_raw, eval_ynorm, filepath="eval_comparison.json"):
    """Speichert beide Evaluationen strukturiert zum Vergleich"""
    results = []
    
    for cat in CATEGORIES:
        for v in VARIANTS:
            col_raw = eval_raw[f"{v}_{cat}_hit"]
            col_ynorm = eval_ynorm[f"{v}_{cat}_hit"]
            
            results.append({
                "variant": v,
                "category": cat,
                "raw_true": col_raw.value_counts()[True],
                "ynorm_true": col_ynorm.value_counts()[True],
                "raw_false": col_raw.value_counts()[False],
                "ynorm_false": col_ynorm.value_counts()[False],
                "abs_improvmnt": col_ynorm.value_counts()[True] - col_raw.value_counts()[True],
                "raw_quota": round(col_raw.mean() * 100, 2),
                "ynorm_quota": round(col_ynorm.mean() * 100, 2),
                "delta_quota": round((col_ynorm.mean() - col_raw.mean()) * 100, 2)
            })
    
    df_results = pd.DataFrame(results)
    df_results.to_json(filepath, orient="records", indent=2)
    df_results.to_csv(filepath.replace(".json", ".csv"), index=False)
    
    return df_results

# Speichern und anzeigen
comparison = save_eval_comparison(just_eval, ynorm)
print(comparison)


   variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       t1    value      2078        2084        130          124   
1       t2    value      2087        2100        121          108   
2       m1    value      1607        1587        601          621   
3       m2    value      1923        1920        285          288   
4       i1    value      1874        1884        334          324   
5       i2    value      1858        1870        350          338   
6       t1     unit      1952        1965        256          243   
7       t2     unit      1935        1956        273          252   
8       m1     unit      1764        1755        444          453   
9       m2     unit      1787        1786        421          422   
10      i1     unit      1761        1777        447          431   
11      i2     unit      1744        1760        464          448   

    abs_improvmnt  raw_quota  ynorm_quota  delta_quota  
0               6      94.11        94.38    

In [70]:
comparison_exact = save_eval_comparison(just_eval_exact, ynorm_exact)
print(comparison_exact)

   variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       t1    value      2057        2063        151          145   
1       t2    value      2056        2069        152          139   
2       m1    value      1607        1587        601          621   
3       m2    value      1909        1906        299          302   
4       i1    value      1851        1845        357          363   
5       i2    value      1820        1816        388          392   
6       t1     unit      1943        1956        265          252   
7       t2     unit      1926        1947        282          261   
8       m1     unit      1755        1746        453          462   
9       m2     unit      1785        1784        423          424   
10      i1     unit      1755        1771        453          437   
11      i2     unit      1735        1751        473          457   

    abs_improvmnt  raw_quota  ynorm_quota  delta_quota  
0               6      93.16        93.43    

In [74]:
# No merge necessary as the two DFs have the same index
comparison_exact["raw_true_any"] = comparison["raw_true"]
comparison_exact["fewer_hits_raw"] = comparison_exact["raw_true"] - comparison_exact["raw_true_any"]
comparison_exact["ynorm_true_any"] = comparison["ynorm_true"]
comparison_exact["fewer_hits_ynorm"] = comparison_exact["ynorm_true"] - comparison_exact["ynorm_true_any"]
comparison_exact[["variant","category","raw_true","ynorm_true","raw_true_any","fewer_hits_raw","ynorm_true_any","fewer_hits_ynorm"]]

,variant,category,raw_true,ynorm_true,raw_true_any,fewer_hits_raw,ynorm_true_any,fewer_hits_ynorm
0,t1,value,2057,2063,2078,-21,2084,-21
1,t2,value,2056,2069,2087,-31,2100,-31
2,m1,value,1607,1587,1607,0,1587,0
3,m2,value,1909,1906,1923,-14,1920,-14
4,i1,value,1851,1845,1874,-23,1884,-39
5,i2,value,1820,1816,1858,-38,1870,-54
6,t1,unit,1943,1956,1952,-9,1965,-9
7,t2,unit,1926,1947,1935,-9,1956,-9
8,m1,unit,1755,1746,1764,-9,1755,-9
9,m2,unit,1785,1784,1787,-2,1786,-2


* You can see that the hits have dropped in general. Interestingly not all equal, not even for the same model.
* The drops are more evenly distributed for units than for values.


# Stuff for GEPA

In [36]:
just_i = gs_extractions_raw_ynorm[
    ['report_name',
    'status',
    'scopes_present',
    'years_present',
    'year',
    'scope',
    'page',
    'value',
    'unit',
    'unit_normalized',
    'label',
    'value_i1',
    'value_i2',
    'unit_i1',
    'unit_i2',
    'label_i1',
    'label_i2']
]

In [38]:
VARIANTS = ["i1", "i2"]

eval_i, eval_i_expended = eval(just_i)

print_eval(eval_i)

### value ###
i1: True = 1884 || False = 324 || Quota = 85.33%
i2: True = 1870 || False = 338 || Quota = 84.69%

### unit ###
i1: True = 1777 || False = 431 || Quota = 80.48%
i2: True = 1760 || False = 448 || Quota = 79.71%



# Detailed Degradation Analysis (Claude!)

In [32]:
def analyze_degradation_detailed(raw_source, ynorm_source, raw_eval, ynorm_eval, variant, category):
    """Detaillierte Analyse: zeigt Gold-Standard vs. was raw/ynorm extrahiert haben"""

    hit_col = f"{variant}_{category}_hit"
    value_col = f"value_{variant}"
    unit_col = f"unit_{variant}"
    gs_col = category  # "value" or "unit"

    # Rows wo raw=True aber ynorm=False (Verschlechterung)
    degraded_mask = (raw_eval[hit_col] == True) & (ynorm_eval[hit_col] == False)
    degraded_idx = raw_source[degraded_mask].index.tolist()

    results = []
    for idx in degraded_idx:
        gs_val = raw_source.iloc[idx][gs_col]
        raw_val = raw_source.iloc[idx][value_col]
        ynorm_val = ynorm_source.iloc[idx][value_col]
        
        results.append({
            "report_name": raw_source.iloc[idx]["report_name"],
            "year": raw_source.iloc[idx]["year"],
            "scope": raw_source.iloc[idx]["scope"],
            "gs_value": gs_val,
            "raw_extracted": raw_val,
            "ynorm_extracted": ynorm_val,
            "raw_was_list": isinstance(raw_val, list),
            "ynorm_was_list": isinstance(ynorm_val, list),
        })

    return pd.DataFrame(results)


# m1 value: detaillierte Analyse
print("=" * 110)
print("M1 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m1_value_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m1", "value")
print(f"Total degraded: {len(m1_value_details)}\n")
if len(m1_value_details) > 0:
    print(m1_value_details.to_string())
    m1_value_details.to_csv("m1_value_degradations.csv", index=False)
    print("\n✓ Saved to: m1_value_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M2 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m2_value_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m2", "value")
print(f"Total degraded: {len(m2_value_details)}\n")
if len(m2_value_details) > 0:
    print(m2_value_details.to_string())
    m2_value_details.to_csv("m2_value_degradations.csv", index=False)
    print("\n✓ Saved to: m2_value_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M1 UNIT DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m1_unit_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m1", "unit")
print(f"Total degraded: {len(m1_unit_details)}\n")
if len(m1_unit_details) > 0:
    print(m1_unit_details.to_string())
    m1_unit_details.to_csv("m1_unit_degradations.csv", index=False)
    print("\n✓ Saved to: m1_unit_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M2 UNIT DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m2_unit_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m2", "unit")
print(f"Total degraded: {len(m2_unit_details)}\n")
if len(m2_unit_details) > 0:
    print(m2_unit_details.to_string())
    m2_unit_details.to_csv("m2_unit_degradations.csv", index=False)
    print("\n✓ Saved to: m2_unit_degradations.csv")
else:
    print("(No degradations)")

M1 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)
Total degraded: 20

                         report_name  year scope  gs_value  raw_extracted                                      ynorm_extracted  raw_was_list  ynorm_was_list
0                addtech_2022_report  2022     1       NaN            NaN                                               [2041]         False            True
1                addtech_2022_report  2022   2lb       NaN            NaN                                               [3709]         False            True
2                addtech_2022_report  2022   2mb       NaN            NaN                                               [4087]         False            True
3                addtech_2022_report  2022     3       NaN            NaN                                              [22961]         False            True
4   evolution mining ltd_2020_report  2016   2mb       NaN            NaN                                             [472257]         False    

m1 and m2 had indeed extracted values that the gold standard did not aknowledge. Therefore a miss-mapping of the "year"-value, which resolved to "NaN" led to correct retrieval. But the fact that it had found a value, which is now shown, the score went down.